# Multiple Regression
## Predicting with several drivers at once

**The business questions.** A car maker wants to know what really buys fuel economy: weight, engine size, being a hybrid, or something else. An estate agent at a ski resort wants a pricing model that is simple enough to explain and honest about how far to trust it. Both questions have several drivers that act at the same time, and the model has to separate them.

You will fit each model in Python, read it in plain English, and check it against JMP. **Python leads and JMP is the check.** If the two disagree, JMP is right until proven otherwise.

---

### Source and credits

The housing exercise follows **JMP Academic Case Study 024, "Housing Prices: Multiple Regression, Multicollinearity and Model Building"**, by Marlene Smith, University of Colorado Denver Business School (JMP Statistical Discovery LLC, 2025, jmp.com/cases). The housing and fuel efficiency data files come from the same JMP case library. The wording, the code and the figures in this notebook are our own. Where we depart from the case: we use the fuel efficiency file first and the housing file second, we read all prices in thousands of US dollars, and we add checks on influential houses and on the choice between two near-identical location measures that the case does not make.

---

### How to use this notebook

- Run cells in order with **Shift+Enter**. The data files are in `../data/`.
- Copilot is on. Where a cell says *Ask Copilot*, write a short prompt, read what it gives you, run it, and **check the answer against the JMP numbers** shown in the notebook.
- Each part ends with a commentary cell. Write in plain words, as if to a manager.
- Prices in the housing data are in **thousands of US dollars** (a price of 330 means $330,000). Distances are in miles.

---

## Part 1: Demo: fuel efficiency with several drivers

### Step 1: Load and look

Two hundred 2011 US car models. The outcome we care about is `MPG Highway`. We rename the columns once so the code stays short.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats

print(os.listdir("../data"))

In [ ]:
fuel = pd.read_excel("../data/fuelefficiency2011.xlsx")

# Short names make the model code readable
fuel = fuel.rename(columns={
    "Make and Model": "model", "MPG Highway": "mpg", "Curb Weight": "weight",
    "Engine Size (L)": "engine", "Cylinders": "cylinders", "Horsepower": "horsepower",
    "Overall Length": "length", "Max Seating Capacity": "seats", "Torque": "torque",
    "Wheelbase": "wheelbase", "Hybrid?": "hybrid_label"})

print(fuel.shape)
fuel.head()

In [ ]:
# Three small helpers used throughout. Read them once, then use them as given.

def fit_ols(data, y, drivers):
    # Fit y on a list of drivers. The constant is the intercept.
    return sm.OLS(data[y], sm.add_constant(data[drivers])).fit()

def vifs(data, drivers):
    # Variance inflation factor for each driver: how much the others overlap with it
    X = sm.add_constant(data[drivers])
    return pd.Series([variance_inflation_factor(X.values, i + 1) for i in range(len(drivers))], index=drivers)

def stepwise(data, y, candidates, criterion="aic", direction="backward"):
    # Add or drop one driver at a time while the criterion (AIC or BIC) keeps falling
    def score(cols):
        if cols:
            return getattr(fit_ols(data, y, cols), criterion)
        return getattr(sm.OLS(data[y], np.ones(len(data))).fit(), criterion)
    chosen = list(candidates) if direction == "backward" else []
    best = score(chosen)
    while True:
        if direction == "backward":
            trials = [[c for c in chosen if c != d] for d in chosen]
        else:
            trials = [chosen + [d] for d in candidates if d not in chosen]
        if not trials:
            break
        s, t = min(((score(t), t) for t in trials), key=lambda x: x[0])
        if s < best - 1e-9:
            best, chosen = s, t
        else:
            break
    return sorted(chosen), round(best, 1)

A model needs a number for each driver. `Hybrid?` is a label, so we turn it into a 0/1 column, the usual way to feed a yes/no driver into a regression. The code below is what an AI assistant produced for this step.

In [ ]:
# 1 if the car is a hybrid, 0 if not
fuel["hybrid"] = (fuel["hybrid_label"] == "Non hybrid").astype(int)

fuel["hybrid"].value_counts()

### Step 2: Run the models

Start with what you already know from one driver: weight on its own. Then add engine size, then everything. Watch what happens to R-squared, adjusted R-squared and the typical miss (RMSE, in mpg).

In [ ]:
drivers_all = ["weight", "engine", "cylinders", "horsepower", "length", "seats", "torque", "wheelbase", "hybrid"]

models = {
    "Weight only": fit_ols(fuel, "mpg", ["weight"]),
    "Weight and engine": fit_ols(fuel, "mpg", ["weight", "engine"]),
    "All nine drivers": fit_ols(fuel, "mpg", drivers_all),
}

pd.DataFrame({
    name: {"R-squared": m.rsquared, "Adjusted R-squared": m.rsquared_adj, "Typical miss (mpg)": np.sqrt(m.mse_resid)}
    for name, m in models.items()}).T.round(3)

R-squared can only go up when you add a driver, even a useless one. Adjusted R-squared charges a penalty for each extra driver, so it can fall. When the two move apart, the new driver is not earning its place.

In [ ]:
full = models["All nine drivers"]
table = pd.DataFrame({
    "Slope": full.params, "Lower 95%": full.conf_int()[0], "Upper 95%": full.conf_int()[1],
    "p-value": full.pvalues, "VIF": [np.nan] + list(vifs(fuel, drivers_all))})
table.round(4)

### Step 3: Read it in plain English

A slope in a multiple regression means: **the change in mpg for one extra unit of this driver, holding the others constant.** So the weight slope says: two cars with the same engine, cylinders, seats and so on, but one pound heavier, differ by about 0.005 mpg. Per 1,000 lb that is about 5 mpg.

Three things to read in the table above:

1. **Which slopes are real?** A p-value above 0.05 means the data cannot separate that slope from zero *given the other drivers*.
2. **Which slopes changed sign or size compared with the driver on its own?**
3. **Which drivers overlap?** A VIF around 10 or more says the driver is largely a mix of the others. It is a rule of thumb, not a firm rule.

In [ ]:
# Each driver on its own against what it does in the full model
alone = fuel[drivers_all + ["mpg"]].corr()["mpg"].drop("mpg")
compare = pd.DataFrame({"Correlation with mpg, alone": alone, "Slope in the full model": full.params.drop("const"), "p-value in full model": full.pvalues.drop("const")})
compare.round(4)

Two drivers deserve a second look:

- **Cylinders** is negative on its own (more cylinders, fewer mpg) but positive in the full model. More cylinders do not make a car more efficient. The engine size slope has already taken the cost of a big engine, and what is left over is the effect of spreading the same litres across more cylinders. Say it as *holding engine size and weight constant*.
- **Max seating capacity** is nearly flat on its own (-0.19) and strongly positive in the model. Heavier, bigger cars carry more people, which hides the effect. Compare cars of the same weight and engine, and the ones that carry more people are more efficient. This is called a *suppressor* effect. It only appears once weight is in the model.

### Step 4: Which drivers overlap?

Horsepower, torque, overall length and wheelbase each have a strong correlation with mpg on their own. In the full model none of them is significant. They carry much the same information as weight and engine size, so the model cannot tell which one deserves the credit.

In [ ]:
print(vifs(fuel, drivers_all).round(1).sort_values(ascending=False))

# Correlations among the overlapping group
fuel[["horsepower", "torque", "length", "wheelbase", "engine", "weight"]].corr().round(2)

### Step 5: Check against JMP

In JMP, open the fuel efficiency file, then **Analyze > Fit Model** with MPG Highway as Y and the nine drivers as model effects. Hybrid holds words, so JMP treats it as nominal, and you have two routes. Route A: leave it nominal. JMP then reports half the gap between the groups (about 5.07), shows the labels in brackets, and gives it a sign that depends on which level it lists first, so double it (about 10.15) to compare with Python. Route B: make a numeric 0/1 column. Add a formula column that is 1 for Hybrid and 0 otherwise, set it to continuous, and use it in the model. Its slope then reads the same way as in Python (about 10.15). Check the coding is the right way round before you trust the sign. The table below shows the numbers you should see for the slopes, set against what your Python run gave. Compare them row by row.

In [ ]:
jmp_expect = pd.Series({
    "const": 38.26, "weight": -0.0051, "engine": -2.03, "cylinders": 0.913, "horsepower": -0.0062,
    "length": -0.0032, "seats": 0.892, "torque": 0.0068, "wheelbase": 0.0441, "hybrid": 10.15})

check = pd.DataFrame({"Python slope": full.params.round(4), "JMP slope (expect)": jmp_expect})
check["Difference"] = (check["Python slope"] - check["JMP slope (expect)"]).round(4)
check

All rows should match to the rounding shown, except two. Look at the Difference column.

### Find the bug

Two rows do not match (the intercept and one slope), and R-squared, the p-values and every other slope are fine. Something went wrong in a step that ran without an error. Find it, fix it, re-run the notebook from that step, and see the row match.

Write down: what was wrong, how you found it, and why the model's R-squared did not reveal it.

In [ ]:
# Your fix goes here. Re-run the cells above afterwards.

### Step 6: The Hybrid driver, and how much to trust it

Once the coding is right, the Hybrid slope is about +10 mpg: a hybrid gets roughly ten more mpg than a non-hybrid of the same weight, engine and the rest. Its VIF is about 1, so it is not tangled up with the other drivers. But count the hybrids.

In [ ]:
print(fuel.groupby("hybrid_label")["mpg"].agg(["count", "mean"]).round(1))

Five cars. A slope that rests on five cars is real in this file and fragile as a forecast for a car that has not been built yet.

### Step 7: Diagnostics

A model can have a good R-squared and still mislead. Three quick checks: are the misses spread evenly, is any car running the model, and is any car a poor fit?

In [ ]:
infl = full.get_influence()
fuel["fitted"] = full.fittedvalues
fuel["residual"] = full.resid
fuel["leverage"] = infl.hat_matrix_diag             # how unusual the car's driver values are
fuel["cooks_d"] = infl.cooks_distance[0]            # how much the car moves the model
fuel["stud_resid"] = infl.resid_studentized_external  # size of the miss, in standard errors

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(fuel["fitted"], fuel["residual"], s=14)
ax[0].axhline(0, color="grey")
ax[0].set_xlabel("Fitted mpg"); ax[0].set_ylabel("Residual (mpg)"); ax[0].set_title("Residuals against fitted")
ax[1].scatter(fuel["leverage"], fuel["cooks_d"], s=14)
ax[1].set_xlabel("Leverage"); ax[1].set_ylabel("Cook's distance"); ax[1].set_title("Which cars run the model?")
plt.tight_layout(); plt.show()

print("Largest Cook's distance:", fuel.loc[fuel["cooks_d"].idxmax(), "model"], round(fuel["cooks_d"].max(), 2))
print("Cars above 4/n (a prompt to look):", int((fuel["cooks_d"] > 4 / len(fuel)).sum()), "| above 1 (a flag):", int((fuel["cooks_d"] > 1).sum()))
print("Highest leverage:", fuel.loc[fuel["leverage"].idxmax(), "model"], round(fuel["leverage"].max(), 2))
print("Cars with |studentised residual| above 3:", fuel.loc[fuel["stud_resid"].abs() > 3, "model"].tolist())
print("Breusch-Pagan p-value (small = spread not constant):", round(het_breuschpagan(full.resid, full.model.exog)[1], 4))
print("Shapiro p-value for the residuals (large = shape is fine):", round(stats.shapiro(full.resid).pvalue, 2))

Reading it: no car has a Cook's distance above 1, so no single car runs the model. The Prius V has the largest value and the only big miss (it does better than the model expects). The misses fan out a little as mpg rises, which the Breusch-Pagan test picks up, but their shape is fine. Nothing here stops us using the model; it tells us to quote its errors with care.

### Step 8: Build a simpler model

Twelve drivers are rarely better than six. We take two routes and compare. **Route A:** drop the weakest driver by p-value, one at a time, until all that remain are significant. **Route B:** let AIC or BIC decide. Both measure fit against complexity; BIC punishes extra drivers harder.

In [ ]:
keep = drivers_all.copy()
while True:
    m = fit_ols(fuel, "mpg", keep)
    p = m.pvalues.drop("const")
    if p.max() <= 0.05:
        break
    print(f"Drop {p.idxmax():<10} p = {p.max():.3f}")
    keep.remove(p.idxmax())

print("\nLeft with:", keep)
print("R-squared", round(m.rsquared, 3), "| adjusted", round(m.rsquared_adj, 3))

In [ ]:
for criterion in ["aic", "bic"]:
    for direction in ["backward", "forward"]:
        chosen, score = stepwise(fuel, "mpg", drivers_all, criterion, direction)
        print(f"{criterion.upper()} {direction:<8}: {chosen}  ({criterion.upper()} {score})")

With 200 cars and a strong signal, every route lands on the same family of models. AIC keeps wheelbase and BIC drops it. A route that automatically picks drivers is a starting point for a candidate model, and you still decide whether it makes business sense. With a smaller file the routes will not agree so tidily.

### Step 9: Predict, with an interval

A prediction is an average for cars like this one. There are two intervals. The **confidence interval** is for the average mpg of all such cars. The **prediction interval** is for one particular car, and it is always wider.

In [ ]:
final_fuel = fit_ols(fuel, "mpg", keep)

# A hypothetical car: 3,500 lb, 2.5 L, 4 cylinders, 5 seats, 105 in wheelbase, not a hybrid
new_car = pd.DataFrame({"const": [1.0], "weight": [3500], "engine": [2.5], "cylinders": [4], "seats": [5], "wheelbase": [105], "hybrid": [0]})
pred = final_fuel.get_prediction(new_car[["const"] + keep]).summary_frame(alpha=0.05)
print("Predicted mpg:", round(pred["mean"].iloc[0], 1))
print("Average car like this, 95% interval:", round(pred["mean_ci_lower"].iloc[0], 1), "to", round(pred["mean_ci_upper"].iloc[0], 1))
print("One car like this, 95% interval:", round(pred["obs_ci_lower"].iloc[0], 1), "to", round(pred["obs_ci_upper"].iloc[0], 1))

### Try this

Change the list below, re-run, and watch what moves. Then write what you learned in the commentary cell.

1. Drop `torque` and `length` from the list and compare the slopes and VIFs with the full model.
2. Add `hybrid` to a model that only has `weight` and `engine`. What happens to the weight slope?

In [ ]:
try_drivers = ["weight", "engine", "hybrid"]   # change this list, then re-run

m = fit_ols(fuel, "mpg", try_drivers)
print(m.params.round(4))
print("Adjusted R-squared:", round(m.rsquared_adj, 3))
print("VIFs:", vifs(fuel, try_drivers).round(1).to_dict())

### Commentary cell 1

1. In your own words, what does "holding the others constant" mean for the weight slope?
2. Pick one driver whose slope changed sign or size between the correlation and the full model. Why did it change?
3. Which drivers overlap, and what did that do to their p-values?
4. A colleague says "the Hybrid slope proves a hybrid version of our heaviest model would gain 10 mpg." What do you tell them?

*Your commentary here*

### Say it to a manager

Two sentences, in business language, from what you found. Name the drivers that matter, give the size of the effect with its units, and say one thing the data cannot tell the manager.

*Your two sentences here*

### Stretch

A blank space for you to try an idea of your own with Copilot. For example: plot the model's prediction against the real mpg for every car and mark the five hybrids; or find the five cars the model predicts worst and say what they have in common.

In [ ]:
# Stretch

---
## Part 2: Exercise: a pricing model for resort homes

> **Source.** This exercise follows *JMP Academic Case Study 024, "Housing Prices: Multiple Regression, Multicollinearity and Model Building"*, by Marlene Smith, University of Colorado Denver Business School (JMP Statistical Discovery LLC, 2025). The data come from the case library. The questions, code and wording here are our own. We use a different order of data (fuel efficiency first), and add checks on influential houses and on a tie between two location measures that the case does not make.

**The business question.** An estate agent at a US ski resort sold 45 homes in 12 months. She wants a pricing model that is simple enough to explain to a client, honest about how far to trust it, and able to point at homes that look priced out of line.

**Units.** Price is in **thousands of US dollars**. Distances are in miles. `DoM` is days on the market.

### Step 1: Load and look

In [ ]:
homes = pd.read_excel("../data/housingprices.xlsx")
homes = homes.rename(columns={"Price": "price", "Beds": "beds", "Baths": "baths", "Square Feet": "sqft",
                              "Miles to Resort": "resort", "Miles to Base": "base", "Acres": "acres",
                              "Cars": "cars", "Years Old": "age", "DoM": "dom"})
print(homes.shape, "| missing values:", int(homes.isna().sum().sum()))
homes.describe().round(1)

### Step 2: Three size drivers, alone and together

Beds, baths and square feet all measure how big a house is. Fit each alone, then all three together. Watch the slopes.

In [ ]:
size = ["beds", "baths", "sqft"]   # change this list and re-run

for d in size:
    m = fit_ols(homes, "price", [d])
    print(f"{d:<6} alone: slope {m.params[d]:8.4f}   R-squared {m.rsquared:.3f}")

together = fit_ols(homes, "price", size)
print("\nTogether: R-squared", round(together.rsquared, 3))
pd.DataFrame({"Slope": together.params, "p-value": together.pvalues}).round(4)

In [ ]:
# The size drivers are correlated with each other about as strongly as with price
homes[["price"] + size].corr().round(3)

Each slope collapses when the other two are in the model: beds falls from 77 to about 18, and square feet from 0.135 to about 0.02. They were sharing the credit for the same thing, how big the house is.

### Step 3: Two location drivers

Miles to the resort and miles to the base are almost the same measurement: the resort and the base are close together.

In [ ]:
print("Correlation between the two:", round(homes["resort"].corr(homes["base"]), 3))
for d in ["resort", "base"]:
    m = fit_ols(homes, "price", [d])
    print(f"{d:<7} alone: slope {m.params[d]:7.3f}   R-squared {m.rsquared:.3f}")

pair = fit_ols(homes, "price", ["resort", "base"])
pd.DataFrame({"Slope": pair.params, "p-value": pair.pvalues}).round(4)

Alone, both are negative: further from the lifts, a cheaper house. Together, resort distance turns **positive** (the wrong sign, and not significant), and base distance takes the whole effect. A sign that flips when a near-copy of the driver enters the model is the classic warning of overlap.

### Step 4: The full model and the VIFs

In [ ]:
all_nine = ["beds", "baths", "sqft", "resort", "base", "acres", "cars", "age", "dom"]
full_h = fit_ols(homes, "price", all_nine)
print("R-squared", round(full_h.rsquared, 3), "| adjusted", round(full_h.rsquared_adj, 3), "| typical miss", round(np.sqrt(full_h.mse_resid), 1), "($000s)")
pd.DataFrame({"Slope": full_h.params, "p-value": full_h.pvalues, "VIF": [np.nan] + list(vifs(homes, all_nine))}).round(3)

Only baths and acres are significant in the nine-driver model, even though the model explains 81% of the variation in price. The two highest VIFs, about 14, belong to the two location measures.

### Step 5: Fix the overlap, then reduce

Drop one of the two near-identical drivers (we drop `base`, as the case does, because the resort is the draw). Then drop the weakest remaining driver by p-value, one at a time, until all are significant.

In [ ]:
keep_h = [d for d in all_nine if d != "base"]
m = fit_ols(homes, "price", keep_h)
print("Without base: R-squared", round(m.rsquared, 3), "| adjusted", round(m.rsquared_adj, 3), "\n")

while True:
    m = fit_ols(homes, "price", keep_h)
    p = m.pvalues.drop("const")
    if p.max() <= 0.05:
        break
    print(f"Drop {p.idxmax():<7} p = {p.max():.3f}")
    keep_h.remove(p.idxmax())
final_h = m
print("\nFinal drivers:", keep_h)

In [ ]:
summary = pd.DataFrame({"Slope": final_h.params, "Lower 95%": final_h.conf_int()[0], "Upper 95%": final_h.conf_int()[1], "p-value": final_h.pvalues})
print(summary.round(4))
print("\nR-squared", round(final_h.rsquared, 3), "| adjusted", round(final_h.rsquared_adj, 3), "| typical miss", round(np.sqrt(final_h.mse_resid), 1), "($000s)")
print("AIC", round(final_h.aic, 1), "| BIC", round(final_h.bic, 1))

In business terms (prices are in $000s): each extra bathroom goes with about $65,600 on the price, each extra square foot about $54, each mile further from the resort about $3,600 less, each extra acre about $5,400 more, all holding the other three constant. The typical miss is about $62,000.

### Step 6: Predict with an interval

Price a house with 2,000 square feet, 2 baths, 15 miles from the resort and 1 acre. Give the estate agent a price for the average house like this and a range for one house.

*Ask Copilot*: write a short prompt that asks for the predicted price and both 95% intervals from the final model. Paste its code below, run it, and check against JMP.

In [ ]:
# The house to price: 2,000 sq ft, 2 baths, 15 miles from the resort, 1 acre
new_home = pd.DataFrame({"const": [1.0], "baths": [2], "sqft": [2000], "resort": [15], "acres": [1]})

In [ ]:
# Paste Copilot's code here, then run it

**JMP check.** In JMP, fit the final model and use the red triangle options to save the prediction formula, the mean confidence interval and the individual confidence interval for a new row. You should see a predicted price of about **358.2**, an interval for the average such house of about **334.1 to 382.3**, and for one such house about **229.9 to 486.4**. If your Python does not match, find out why before you move on.

### Step 7: Python-only extras

JMP's menus can do most of this, but Python makes it quick to ask a different question. Three questions follow.

**Question A. Do the automatic routes pick the same model?**

In [ ]:
for criterion in ["aic", "bic"]:
    for direction in ["backward", "forward"]:
        chosen, score = stepwise(homes, "price", all_nine, criterion, direction)
        print(f"{criterion.upper()} {direction:<8}: {chosen}  ({criterion.upper()} {score})")

with_base = fit_ols(homes, "price", ["baths", "sqft", "base", "acres"])
print("\nResort model adjusted R-squared:", round(final_h.rsquared_adj, 3), "| Base model:", round(with_base.rsquared_adj, 3))

All four routes choose **miles to base**, not miles to the resort, by a hair. The two measures carry almost the same information, so the data cannot decide between them. The case chose the resort on business judgement (the resort is the attraction). The algorithm prefers the base. When a tie like this appears, the choice belongs to a person who knows the business.

**Question B. Does one house run the model?**

In [ ]:
infl_h = final_h.get_influence()
homes["leverage"] = infl_h.hat_matrix_diag
homes["cooks_d"] = infl_h.cooks_distance[0]
homes["stud_resid"] = infl_h.resid_studentized_external
homes.index = homes.index + 1   # count rows from 1, as JMP does

print(homes.sort_values("cooks_d", ascending=False)[["price", "sqft", "resort", "acres", "leverage", "cooks_d", "stud_resid"]].head(3).round(2))
print("\nLargest lots (acres):", homes["acres"].sort_values(ascending=False).head(4).tolist())

In [ ]:
# Refit without the house that has the largest Cook's distance
worst = homes["cooks_d"].idxmax()
without = fit_ols(homes.drop(worst), "price", keep_h)

pd.DataFrame({"With all 45 houses": final_h.params, "Without that house": without.params}).round(4)

In [ ]:
print("Adjusted R-squared:", round(final_h.rsquared_adj, 3), "->", round(without.rsquared_adj, 3))
print("Typical miss ($000s):", round(np.sqrt(final_h.mse_resid), 1), "->", round(np.sqrt(without.mse_resid), 1))
print("Houses with more than 5 acres:", int((homes["acres"] > 5).sum()))

pred2 = without.get_prediction(new_home[["const"] + keep_h]).summary_frame()
print("Prediction for the 2,000 sq ft house without it:", round(pred2["mean"].iloc[0], 1))

One remote 40-acre property (priced at 230, with a Cook's distance well above 1) carries the acre slope. Without it, an acre is worth about $12,000, not $5,400, and the model fits better. The prediction for an ordinary house barely moves. So the headline "an acre adds $5,352" depends on one unusual property. **Look at the house, do not delete it silently, and report both models.**

**Question C. Which houses look priced out of line?**

In [ ]:
flagged = homes[homes["stud_resid"].abs() > 2][["price", "beds", "baths", "sqft", "resort", "acres", "stud_resid"]]
flagged.sort_values("stud_resid").round(2)

A studentised residual beyond 2 in size is a miss worth looking at; beyond 3 is a big one. A large positive miss means the house sold for more than the model expects (a possible overpriced listing). A large negative miss means less than expected (a possible bargain). Look at the 230 property again: it is also the house the model has no business pricing, a remote acreage far outside the rest of the data. A bargain finder built on this model would flag it and be wrong to call it a bargain.

Last, the spread of the misses.

In [ ]:
print("Breusch-Pagan p-value:", round(het_breuschpagan(final_h.resid, final_h.model.exog)[1], 3))
print("Shapiro p-value for the residuals:", round(stats.shapiro(final_h.resid).pvalue, 2))

The spread of misses is not quite constant (a small p-value), and the shape of the misses is fine. With only 45 houses, the conclusion on spread is weak. A larger caution: we chose the drivers by dropping the weakest one at a time, using the same 45 houses that we then judged the model on. The p-values and R-squared are therefore more flattering than the model will deserve on new houses. The way to test that honestly is to hold some houses back, which honest testing on unseen data does.

### Try this

Point the same code at a different question. Fit `price` on `sqft` alone, then `sqft` and `acres`, then `sqft`, `acres` and `age`. Which slope moves most, and why? Write what you notice in the commentary cell.

In [ ]:
try_h = ["sqft", "acres"]   # change this list, then re-run

m = fit_ols(homes, "price", try_h)
print(m.params.round(4))
print("Adjusted R-squared:", round(m.rsquared_adj, 3), "| VIFs:", vifs(homes, try_h).round(1).to_dict())

### Commentary cell 2

1. Which two drivers overlapped most strongly in the housing file, and what did that do to the slopes?
2. The case drops miles to base. The automatic routes prefer it. Who decides, and on what grounds?
3. The headline "each acre adds $5,352" depends on one property. How would you present the acre effect to a client?
4. Would you use this model to price houses at the resort next year? What would you want first?

*Your commentary here*

### Say it to a manager

Two sentences, in business language, for the estate agent. Name what drives the price, give one effect with its units, and say how far to trust the model.

*Your two sentences here*

### Should you launch, change or stop?

The estate agent is considering a "bargain finder" that flags every listing the model prices well below the asking price. Write three sentences: would you **launch** it, **change** it, or **stop** it, and what would it cost to be wrong? Think about what the finder did with the 40-acre property.

*Your answer here*

### Stretch

A blank space for your own idea. For example: plot the final model's predicted prices against actual prices and mark the flagged houses; or find which house is the best candidate for a genuine bargain once you set the 40-acre property aside.

In [ ]:
# Stretch